## Task 1.1 — Your first notebook
Create a new notebook, add a title cell, and run the standard imports from the
Introduction. Then generate and plot the elementary sequences below.
The expected result corresponds to handout Figure 1.4 (which shows five
sequences; yours shows four).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import wavfile

n = np.arange(-5, 11)
delta = (n == 0).astype(float)       # unit impulse
step  = (n >= 0).astype(float)       # unit step
ramp  = n * step                     # unit ramp
expo  = 0.8**n * step                # real exponential

fig, axes = plt.subplots(1, 4, figsize=(13, 2.6))
for ax, x, title in zip(axes, [delta, step, ramp, expo], ["δ(n)", "u(n)", "r(n)", "(0.8)^n u(n)"]):
    ax.stem(n, x, basefmt="k-")
    ax.set_title(title); ax.set_xlabel("n")
plt.tight_layout(); plt.show()

## Task 1.2 — Change the parameters

**(a)** Plot $a^n u(n)$ for $a = 0.5$, $a = -0.8$ and $a = 1.1$ on one
figure using subplots.

**(b)** Plot $\cos(2\pi n/8)$ and $\cos(2\pi n/7.5)$ for $0 \le n \le 40$.
Which one repeats exactly? (You will explain why in Week 3.)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from fractions import Fraction
from math import gcd

n = np.arange(0, 41)          # 41 samples: 0 <= n <= 40

# ------------------------------------------------------------------
# (a) Exponential sequences x(n) = a^n u(n)
# ------------------------------------------------------------------
a_values = [0.5, -0.8, 1.1]
behaviour = ["decay to 0", "alternating decay", "grows without bound"]

fig, axes = plt.subplots(3, 1, figsize=(9, 8), sharex=True)
for ax, a, desc in zip(axes, a_values, behaviour):
    x = (a ** n) * (n >= 0)               # u(n) = 1 for n >= 0
    ax.stem(n, x, basefmt="k-")
    ax.set_title(f"x(n) = ({a})^n u(n)  —  {desc}")
    ax.set_ylabel("x(n)")
    ax.grid(True, alpha=0.3)
axes[-1].set_xlabel("n")
fig.suptitle("Task 1.2(a): Effect of the parameter a", y=1.01)
plt.tight_layout()
plt.show()

# ------------------------------------------------------------------
# (b) Discrete-time cosines cos(2*pi*n / M)
# ------------------------------------------------------------------
M_values = [8, 7.5]

def period_from_M(M):
    # Smallest integer period of cos(2*pi*n/M)
    ratio = 1 / Fraction(str(M))          # Omega/(2*pi) = 1/M
    p, q = ratio.numerator, ratio.denominator
    return q // gcd(p, q)

cosines = []
for M in M_values:
    Omega = 2 * np.pi / M
    N = period_from_M(M)
    x = np.cos(Omega * n)
    cosines.append((M, Omega, N, x))
    max_error = np.max(np.abs(x[:-N] - x[N:]))
    print(f"M = {M:>4}: Omega/2pi = {1/M:.6f}, period N = {N:>2}, "
          f"max |x(n)-x(n+N)| = {max_error:.2e}")

fig, axes = plt.subplots(2, 1, figsize=(9, 5), sharex=True)
for ax, (M, Omega, N, x) in zip(axes, cosines):
    ax.stem(n, x, basefmt="k-")
    ax.set_title(f"cos(2*pi*n/{M})  —  repeats every N = {N} samples")
    ax.set_ylabel("x(n)")
    ax.grid(True, alpha=0.3)
axes[-1].set_xlabel("n")
fig.suptitle("Task 1.2(b): Two discrete-time cosines", y=1.01)
plt.tight_layout()
plt.show()

**Answer for Task 1.2 — discussion**

**(a)** The parameter $a$ controls the exponential $x(n)=a^n u(n)$:

- $a=0.5$: $|a|<1$ and positive → smooth decay to 0.
- $a=-0.8$: $|a|<1$ and negative → magnitude decays but the sign alternates each sample.
- $a=1.1$: $|a|>1$ → the sequence grows without bound (this is why $h(n)=1.1^n u(n)$ cannot be a stable impulse response — see Analysis Q3).

**(b)** For $x(n)=\cos(\Omega n)$ with $\Omega=\frac{2\pi}{M}$, the sequence is periodic
only if $\frac{\Omega}{2\pi} = \frac{1}{M}$ is rational. The integer period is found from
$\Omega N = 2\pi k$.

- $M=8$: $\frac{1}{M} = \frac{1}{8}$, so $N=8$. Check: $x(n+8)=x(n)$.
- $M=7.5 = 15/2$: $\Omega = \frac{4\pi}{15}$, $\frac{1}{M} = \frac{2}{15}$, so $2N = 15k$ and the
  smallest integer solution is $N=15$. Check: $x(n+15)=x(n)$.

**Both sequences repeat exactly**; the first every 8 samples, the second every 15
samples (not 7.5). The code verifies this numerically by comparing $x(n)$ with
$x(n+N)$ over the plotted range; the maximum error is at machine precision.
Week 3 will explain the general rule in more detail.

## Task 1.3 — Working with audio

Record 3–5 seconds of your voice with your phone, save it as a WAV file and
upload it to Colab (Files panel). Set `fname` to your file name. If the file is
not present, the fallback test signal is used so the code still runs.

In [ ]:
import os
from scipy.io import wavfile
import numpy as np
import matplotlib.pyplot as plt

fname = "my_voice.wav"                 # change to your file name
if os.path.exists(fname):
    fs, x = wavfile.read(fname)
    if x.ndim > 1:                          # stereo -> mono
        x = x.mean(axis=1)
    x = x / np.max(np.abs(x))               # normalise to [-1, 1]
else:                                       # fallback test signal
    fs = 16000
    t = np.arange(0, 2, 1/fs)
    x = 0.5*np.sin(2*np.pi*440*t) * np.exp(-t)

t = np.arange(len(x)) / fs
print(f"fs = {fs} Hz, duration = {len(x)/fs:.2f} s, samples = {len(x)}")
plt.plot(t, x); plt.xlabel("time (s)"); plt.ylabel("amplitude")
plt.title("My recording"); plt.show()

To listen to the signal in the notebook, run:

In [ ]:
from IPython.display import Audio
Audio(x, rate=fs)

## Checkpoint 1
Show the instructor your plots from **Tasks 1.1–1.3** and the printed
**sampling rate** and **duration** of your recording.

## Analysis questions

**1. What is the sampling rate of your phone's recording? How many numbers are
needed to store one minute of it in mono?**

From the output of Task 1.3 with my uploaded `my_voice.wav`:

- Sampling rate: $f_s = 48{,}000$ Hz
- Duration: $3.71$ s
- Samples in the clip: $178{,}176$  
  (Note: $\frac{178{,}176}{48{,}000} = 3.712$ s, which the code prints rounded to 3.71 s.)

One minute of mono audio needs one number (sample) per sampling instant:

$$
N = f_s \times 60 = 48{,}000 \times 60 = 2{,}880{,}000 \text{ numbers}.
$$

For reference, my 3.71 s clip itself contains 178,176 numbers. If each sample is
stored as a 16-bit (2-byte) integer, one minute requires
$2{,}880{,}000 \times 2 = 5{,}760{,}000$ bytes (about 5.76 MB, or 5.49 MiB).

**2. Why is a stem plot more appropriate than a continuous line for $x(n)$?**

$x(n)$ is defined **only at integer values of $n$**. A stem plot draws a
line at each integer index and leaves the gaps empty, showing exactly the
available samples. A continuous line would interpolate between integer points
and wrongly suggest that the signal has values at non-integer $n$, which it
does not.

**3. For $a = 1.1$ the exponential grows. Which property of a system would such
a signal violate if it were an impulse response (Unit 1 §1.7)?**

It violates **BIBO stability**. An LTI system is Bounded-Input Bounded-Output
stable only if its impulse response is absolutely summable,
$\sum_n |h(n)| < \infty$. With $h(n) = 1.1^n u(n)$, the terms grow and the
sum diverges, so a bounded input can produce an unbounded output. A stable
system's impulse response must decay, not grow.